In [ ]:
import numpy as np
import geopandas as gpd
import os
from pynhd import NLDI
import dataretrieval.nwis as nwis
import pandas as pd
import rasterio
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import rioxarray as rxr
import shapely
from shapely.geometry import mapping

# Create all Basin Shapefiles in gage_ids.csv

I used they USGS python package NLDI Flow Tools to easily dilineate basin off gage id
https://www.usgs.gov/software/nldi-flow-tools

In [ ]:
def delineate_basin_from_gage(
    gage_id,
    out_dir=r"C:\Repos\EF5-RTI\control_files\data\basin_dilineations"
):
    """
    Delineate an upstream basin polygon from a USGS stream gage ID.

    Parameters
    ----------
    gage_id : str
        USGS site number, e.g. "06752260"
    out_dir : str
        Folder to save output files

    Returns
    -------
    basin_gdf : geopandas.GeoDataFrame
        Upstream basin polygon
    site_gdf : geopandas.GeoDataFrame
        Gage point
    """

    os.makedirs(out_dir, exist_ok=True)

    # Make sure gage_id is a clean string
    gage_id = str(gage_id).strip()

    # 1) Pull site metadata from USGS NWIS
    site = nwis.get_info(sites=gage_id)[0]

    if site.empty:
        raise ValueError(f"No USGS site metadata found for gage {gage_id}")

    # USGS site metadata commonly contains decimal latitude/longitude
    lat = float(site.iloc[0]["dec_lat_va"])
    lon = float(site.iloc[0]["dec_long_va"])

    # Build gage point GeoDataFrame
    site_gdf = gpd.GeoDataFrame(
        site.copy(),
        geometry=gpd.points_from_xy([lon], [lat]),
        crs="EPSG:4326",
    )

    # 2) Delineate basin using NLDI / PyNHD
    nldi = NLDI()

    # Local catchment
    try:
        split_catchment = nldi.get_basins(
            [gage_id],
            fsource="nwissite",
            split_catchment=True,
            simplified=False,
        )
    except Exception:
        split_catchment = None

    # Full upstream basin
    basin_gdf = nldi.get_basins(
        [gage_id],
        fsource="nwissite",
        split_catchment=False,
        simplified=False,
    )

    if basin_gdf is None or basin_gdf.empty:
        raise ValueError(
            f"NLDI did not return a basin for gage {gage_id}. "
            "This can happen for some sites or locations outside supported coverage."
        )

    # Save outputs
    basin_path = os.path.join(out_dir, f"{gage_id}_basin.geojson")
    site_path = os.path.join(out_dir, f"{gage_id}_gage.geojson")
    local_path = os.path.join(out_dir, f"{gage_id}_localcatchment.geojson")

    basin_gdf.to_file(basin_path, driver="GeoJSON")
    site_gdf.to_file(site_path, driver="GeoJSON")

    if split_catchment is not None and not split_catchment.empty:
        split_catchment.to_file(local_path, driver="GeoJSON")

    return basin_gdf, site_gdf

## Just set path to csv with gage ids (make sure column with gage ids is "gage_id")

In [ ]:
gage_ids = pd.read_csv(
    r"C:\Repos\EF5-RTI\control_files\data\gage_ids.csv",
    dtype=str
).iloc[:, 0].tolist()

for gage_id in gage_ids:
    try:
        delineate_basin_from_gage(gage_id)
        print(f"Done: {gage_id}")
    except Exception as e:
        print(f"Failed: {gage_id} -> {e}")

# Clip Basic Files using Basin Shapefile

In [ ]:
import os
from pathlib import Path

import geopandas as gpd
import rasterio
from rasterio.mask import mask


# =============================================================================
# USER INPUTS
# =============================================================================
basin_name = "South_Toe_03463300"

basin_boundary = r"C:\Repos\EF5-RTI\Data\basin_dilineations\03463300_basin.geojson"

# These file paths should't need to be changed when clipping basin within CONUS
flow_direction_raster = r"C:\Repos\EF5-RTI\Data\EF5_US_Params\basic\fdir_usa.tif"
flow_accumulation_raster = r"C:\Repos\EF5-RTI\Data\EF5_US_Params\basic\facc_usa.tif"
dem_raster = r"C:\Repos\EF5-RTI\Data\EF5_US_Params\basic\dem_usa.tif"

crest_param_folder = r"C:\Repos\EF5-RTI\Data\EF5_US_Params\crest_params"
kw_param_folder = r"C:\Repos\EF5-RTI\Data\EF5_US_Params\kw_params"

# Output folder for where clipped rasters will be saved. Subfolders will be created for main rasters, crest params, and kw params
output_base_folder = r"C:\Repos\EF5-RTI\Data\clipped_outputs"


# =============================================================================
# SETUP
# =============================================================================
main_raster_output_folder = os.path.join(output_base_folder, "main_layers")
crest_output_folder = os.path.join(output_base_folder, "crest_parameters")
kw_output_folder = os.path.join(output_base_folder, "kw_parameters")

for folder in [
    output_base_folder,
    main_raster_output_folder,
    crest_output_folder,
    kw_output_folder,
]:
    os.makedirs(folder, exist_ok=True)


# =============================================================================
# LOAD BASIN BOUNDARY
# =============================================================================
if not os.path.exists(basin_boundary):
    raise FileNotFoundError(f"Basin boundary not found: {basin_boundary}")

basin_gdf = gpd.read_file(basin_boundary)

if basin_gdf.empty:
    raise ValueError("Basin boundary file loaded, but it contains no features.")

if basin_gdf.crs is None:
    raise ValueError("Basin boundary has no CRS defined.")


# =============================================================================
# HELPER FUNCTION
# =============================================================================
def clip_raster_to_basin(in_raster: str, basin_feature: gpd.GeoDataFrame, out_raster: str) -> None:
    try:
        print(f"Clipping: {in_raster}")

        with rasterio.open(in_raster) as src:
            # Reproject basin to raster CRS if needed
            if basin_feature.crs != src.crs:
                basin_proj = basin_feature.to_crs(src.crs)
            else:
                basin_proj = basin_feature

            shapes = [geom for geom in basin_proj.geometry if geom is not None and not geom.is_empty]
            if not shapes:
                raise ValueError(f"No valid geometry found for clipping {in_raster}")

            clipped_data, clipped_transform = mask(
                src,
                shapes,
                crop=True,
                nodata=src.nodata,
            )

            out_meta = src.meta.copy()
            out_meta.update(
                {
                    "height": clipped_data.shape[1],
                    "width": clipped_data.shape[2],
                    "transform": clipped_transform,
                }
            )

            # Ensure output folder exists
            os.makedirs(os.path.dirname(out_raster), exist_ok=True)

            with rasterio.open(out_raster, "w", **out_meta) as dst:
                dst.write(clipped_data)

        print(f"Saved: {out_raster}")

    except Exception as e:
        print(f"Failed to clip {in_raster}")
        print(e)


# =============================================================================
# CLIP MAIN RASTERS
# =============================================================================
flow_dir_out = os.path.join(main_raster_output_folder, f"{basin_name}_flow_direction.tif")
flow_acc_out = os.path.join(main_raster_output_folder, f"{basin_name}_flow_accumulation.tif")
dem_out = os.path.join(main_raster_output_folder, f"{basin_name}_dem.tif")

clip_raster_to_basin(flow_direction_raster, basin_gdf, flow_dir_out)
clip_raster_to_basin(flow_accumulation_raster, basin_gdf, flow_acc_out)
clip_raster_to_basin(dem_raster, basin_gdf, dem_out)


# =============================================================================
# CLIP CREST PARAMS
# =============================================================================
crest_files = [f for f in os.listdir(crest_param_folder) if f.lower().endswith(".tif")]
print(f"\nFound {len(crest_files)} CREST parameter rasters.")

for tif_name in crest_files:
    in_raster = os.path.join(crest_param_folder, tif_name)
    out_raster = os.path.join(crest_output_folder, f"{basin_name}_{tif_name}")
    clip_raster_to_basin(in_raster, basin_gdf, out_raster)


# =============================================================================
# CLIP KW PARAMS
# =============================================================================
kw_files = [f for f in os.listdir(kw_param_folder) if f.lower().endswith(".tif")]
print(f"\nFound {len(kw_files)} KW parameter rasters.")

for tif_name in kw_files:
    in_raster = os.path.join(kw_param_folder, tif_name)
    out_raster = os.path.join(kw_output_folder, f"{basin_name}_{tif_name}")
    clip_raster_to_basin(in_raster, basin_gdf, out_raster)


print("\nAll clipping complete.")

# Observations (USGS streamflow)

Optional, but needed for calibration/analysis anyway.

Use `fetch_usgs_from_control.py` to download USGS streamflow for a gauge and date range.

```bash
# From EF5-RTI folder
python3 fetch_usgs_from_control.py \
  --gauge 04085200 \
  --start-date 2022-07-27 \
  --end-date 2022-07-30 \
  --outdir ~/Kewaunee/observations
```

Notes:
- `--gauge` is the USGS site ID.
- `--start-date` and `--end-date` accept `YYYY-MM-DD`, `YYYYMMDDHHMMSS`, or ISO-8601.
- The script writes all available timesteps in the range (no aggregation).
- Output file name format: `Streamflow_Time_Series_CMS_UTC_USGS_<gauge>.csv`.

# Next, download MRMS data using script
Configured for CONUS only, using the Iowa data store.

```bash 
# From EF5-RTI folder
./download_mrms_preciprate.sh -h

Usage: download_mrms_preciprate.sh [OPTIONS]

Download MRMS PrecipRate .gz files from mtarchive and decompress them.

Options:
  -s, --start-date YYYY-MM-DD   Start date (inclusive). Default: 2022-07-27
  -e, --end-date YYYY-MM-DD     End date (inclusive). Default: 2022-07-30
  -d, --dest-dir PATH           Destination directory. Default: ~/MRMS_preciprate
  -n, --dry-run                 Show what would be downloaded/skipped without changes
  -h, --help                    Show this help message

```
Then run the following cell to conver the grib files to tif. Set the path to your files, and output folder where the tif files will be stored. 

# Convert MRMS File Type After Running "download_mrms_preciprate.sh bash code"

This assures that the MRMS data aligns with the coordinate system used for CONUS wide DEMs

In [ ]:
import os
import numpy as np
import xarray as xr
import rasterio
from rasterio.transform import from_origin

path_to_files = r"C:\Repos\EF5-RTI\Precipitation"
output_path = r"C:\Repos\EF5-RTI\South_Toe_03463300\Forcings\precipitation_fixed"

os.makedirs(output_path, exist_ok=True)

missing = -1
no_coverage = -3
nodata_out = -9999

files = [f for f in os.listdir(path_to_files) if f.endswith(".grib2")]

for file_name in files:
    input_file = os.path.join(path_to_files, file_name)
    output_file = os.path.join(output_path, os.path.splitext(file_name)[0] + ".tif")

    print(f"Processing {file_name}...")

    ds = xr.open_dataset(input_file, engine="cfgrib")
    da = ds[list(ds.data_vars)[0]]

    data = da.values.astype("float32")
    data[(data == missing) | (data == no_coverage)] = nodata_out
    data = np.where(np.isnan(data), nodata_out, data)

    lon = ds.longitude.values
    lat = ds.latitude.values

    # Convert longitudes from 0..360 to -180..180
    lon = np.where(lon > 180, lon - 360, lon)

    # Sort longitude so raster writes left-to-right correctly
    sort_idx = np.argsort(lon)
    lon = lon[sort_idx]
    data = data[:, sort_idx]

    # Make sure latitude goes north to south for raster layout
    if lat[0] < lat[-1]:
        lat = lat[::-1]
        data = np.flipud(data)

    xres = float(abs(lon[1] - lon[0]))
    yres = float(abs(lat[1] - lat[0]))

    transform = from_origin(float(lon.min()), float(lat.max()), xres, yres)

    with rasterio.open(
        output_file,
        "w",
        driver="GTiff",
        height=data.shape[0],
        width=data.shape[1],
        count=1,
        dtype="float32",
        crs="EPSG:4269",
        transform=transform,
        nodata=nodata_out,
    ) as dst:
        dst.write(data, 1)

    ds.close()
    print(f"Converted: {output_file}")

## Optional: Check and make sure coordinate systems for MRMS tif files match basic files

In [ ]:
import rasterio

dem_file = r"C:\Repos\EF5-RTI\South_Toe_03463300\BasicData\South_Toe_03463300_dem.tif"
ppt_file = r"C:\Repos\EF5-RTI\South_Toe_03463300\Forcings\Precipitation\PrecipRate_00.00_20250512-000000.tif"
fdir_file = r"C:\Repos\EF5-RTI\South_Toe_03463300\BasicData\South_Toe_03463300_flow_direction.tif"
facc_file = r"C:\Repos\EF5-RTI\South_Toe_03463300\BasicData\South_Toe_03463300_flow_accumulation.tif"

files = {
    "DEM": dem_file,
    "PPT": ppt_file,
    "FDIR": fdir_file,
    "FACC": facc_file,
}

for name, path in files.items():
    with rasterio.open(path) as src:
        print(f"\n--- {name} ---")
        print("Path:", path)
        print("CRS:", src.crs)
        print("Shape:", (src.height, src.width))
        print("Resolution:", src.res)
        print("Bounds:", src.bounds)
        print("Transform:", src.transform)
        print("Dtype:", src.dtypes)
        print("NoData:", src.nodata)

# Visualize Output Data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

csv_file = r"C:\Repos\EF5-RTI\South_Toe_03463300\Output\ts.03463300.crest.csv"

df = pd.read_csv(csv_file)
df.columns = [c.strip() for c in df.columns]
df["Time"] = pd.to_datetime(df["Time"])

df["Discharge(m^3 s^-1)"] = pd.to_numeric(df["Discharge(m^3 s^-1)"], errors="coerce")
df["Observed(m^3 s^-1)"] = pd.to_numeric(df["Observed(m^3 s^-1)"], errors="coerce")
df["Precip(mm h^-1)"] = pd.to_numeric(df["Precip(mm h^-1)"], errors="coerce")

# set time index so interpolation uses actual timestamps
df = df.set_index("Time")

# interpolate observed values for plotting
df["Observed_interp"] = df["Observed(m^3 s^-1)"].interpolate(method="time")

fig, ax1 = plt.subplots(figsize=(15, 7))

ax1.plot(
    df.index,
    df["Discharge(m^3 s^-1)"],
    label="Simulated Discharge",
    linewidth=2
)

ax1.plot(
    df.index,
    df["Observed_interp"],
    label="Observed Streamflow",
    linewidth=2,
    linestyle="--"
)

ax1.set_ylabel("Streamflow (m$^3$ s$^{-1}$)", fontsize=12)
ax1.grid(True, linestyle="--", alpha=0.4)

ax2 = ax1.twinx()

if len(df) > 1:
    dt_days = (df.index[1] - df.index[0]).total_seconds() / 86400
    bar_width = dt_days * 0.9
else:
    bar_width = 0.01

ax2.bar(
    df.index,
    df["Precip(mm h^-1)"],
    width=bar_width,
    alpha=0.35,
    label="Precipitation"
)

ax2.set_ylabel("Precipitation (mm h$^{-1}$)", fontsize=12)
ax2.invert_yaxis()

ax1.xaxis.tick_top()
ax1.xaxis.set_label_position("top")
ax1.xaxis.set_major_locator(mdates.AutoDateLocator())
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d\n%H:%M"))

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

plt.title("Observed vs Simulated Streamflow with Precipitation", fontsize=14, pad=20)
plt.tight_layout()
plt.show()